<a href="https://colab.research.google.com/github/LuizFelipeHilgert/kubernetes-scheduler/blob/main/desafio2_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import keras
from keras.callbacks import ModelCheckpoint

import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import os

import kagglehub

In [4]:
path = kagglehub.dataset_download(
    "rahultp97/louisiana-flood-2016"
)

print(path)

Using Colab cache for faster access to the 'louisiana-flood-2016' dataset.
/kaggle/input/louisiana-flood-2016


In [9]:
train_df = pd.read_csv(path + '/train.csv')
test_df = pd.read_csv(path + '/test.csv')


In [10]:
import tensorflow as tf
import os

image_size = (512, 360)
batch_size = 32

def preprocess_image_py(image_path_tensor, label):
    image_path = image_path_tensor.numpy().decode('utf-8')
    try:
        img = tf.io.read_file(image_path)
        img = tf.image.decode_image(img, channels=3)
        if img.shape.rank == 0:  # Verifica se a imagem está vazia
            raise ValueError(f"imagem vazia")
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32) / 255.0  # Normaliza para o intervalo [0, 1]
        return img, label
    except Exception as e:
        print(f"Erro ao processar a imagem {image_path}: {e}")
        # Retorna uma imagem preta preenchida com zeros como substituta
        return tf.zeros((*image_size, 3), dtype=tf.float32), label

def preprocess_image(image_path, label):
    processed_image, processed_label = tf.py_function(
        preprocess_image_py,
        inp=[image_path, label],
        Tout=[tf.float32, label.dtype]
    )
    processed_image.set_shape([*image_size, 3])
    processed_label.set_shape([])
    return processed_image, processed_label

# --- Criar Dataset de Treino ---
print("Criando dataset de treino...")
train_image_filenames = train_df['Image ID'].values
train_labels = train_df['Flooded'].values

train_image_paths = [os.path.join(path, 'train', fname) for fname in train_image_filenames]
train_dataset = tf.data.Dataset.from_tensor_slices((train_image_paths, train_labels))

train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=len(train_image_paths)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# --- Criar Dataset de Teste ---
print("Criando dataset de teste...")
test_image_filenames = test_df['Image ID'].values
test_labels = test_df['Flooded'].values

test_image_paths = [os.path.join(path, 'test', fname) for fname in test_image_filenames]
test_dataset = tf.data.Dataset.from_tensor_slices((test_image_paths, test_labels))
test_dataset = test_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE) # Não embaralha o conjunto de teste

print("Datasets de treino e teste criados com sucesso.")
print(f"Número de amostras de treino: {len(train_image_paths)}")
print(f"Número de amostras de teste: {len(test_image_paths)}")

# Opcional: Verificar um batch do dataset de treino
for images, labels in train_dataset.take(1):
    print(f"Formato do batch de imagens de treino: {images.shape}")
    print(f"Formato do batch de labels de treino: {labels.shape}")
    print(f"Tipo de dados do batch de imagens de treino: {images.dtype}")
    print(f"Tipo de dados do batch de labels de treino: {labels.dtype}")
    break


Criando dataset de treino...
Criando dataset de teste...
Datasets de treino e teste criados com sucesso.
Número de amostras de treino: 270
Número de amostras de teste: 52
Formato do batch de imagens de treino: (32, 512, 360, 3)
Formato do batch de labels de treino: (32,)
Tipo de dados do batch de imagens de treino: <dtype: 'float32'>
Tipo de dados do batch de labels de treino: <dtype: 'int64'>


In [11]:
from keras.applications import MobileNetV2
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.optimizers import Adam

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(image_size[0], image_size[1], 3)
)

# Congelar as camadas do modelo base
for layer in base_model.layers:
    layer.trainable = False

# Camadas para classificação binária
x = GlobalAveragePooling2D()(base_model.output)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(
    0.5
)(x)

output_layer = Dense(
    1,
    activation='sigmoid'
)(x)

# Cria o modelo final
model = Model(
    inputs=base_model.input,
    outputs=output_layer
)

# Compila o modelo
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Exibe o resumo do modelo
model.summary()

print("Modelo MobileNetV2 criado e compilado com sucesso!")

/tmp/ipykernel_14415/787230319.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 360,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 256, 180,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 256, 180,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 256, 180,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 180,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 180,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 256, 180,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 256, 180,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 256, 180,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 256, 180,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 256, 180,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 256, 180,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 257, 181,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 128, 90,   │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 128, 90,   │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 128, 90,   │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 128, 90,   │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Modelo MobileNetV2 criado e compilado com sucesso!


In [19]:
import time

checkpointer = ModelCheckpoint(
    filepath='model.weights.best.keras',
    verbose=1,
    save_best_only=True
)

inicio = time.time()

hist = model.fit(
    train_dataset,
    epochs=50,
    validation_data=test_dataset,
    callbacks=[checkpointer],
    verbose=2
)

fim = time.time()
tempo_total = fim - inicio

Epoch 1/50

Epoch 1: val_loss improved from None to 0.10173, saving model to model.weights.best.keras

Epoch 1: finished saving model to model.weights.best.keras
9/9 - 7s - 749ms/step - accuracy: 0.9815 - loss: 0.0640 - val_accuracy: 0.9615 - val_loss: 0.1017
Epoch 2/50

Epoch 2: val_loss improved from 0.10173 to 0.09940, saving model to model.weights.best.keras

Epoch 2: finished saving model to model.weights.best.keras
9/9 - 5s - 570ms/step - accuracy: 0.9889 - loss: 0.0602 - val_accuracy: 0.9615 - val_loss: 0.0994
Epoch 3/50

Epoch 3: val_loss improved from 0.09940 to 0.09682, saving model to model.weights.best.keras

Epoch 3: finished saving model to model.weights.best.keras
9/9 - 5s - 538ms/step - accuracy: 0.9926 - loss: 0.0524 - val_accuracy: 0.9615 - val_loss: 0.0968
Epoch 4/50

Epoch 4: val_loss improved from 0.09682 to 0.09531, saving model to model.weights.best.keras

Epoch 4: finished saving model to model.weights.best.keras
9/9 - 4s - 441ms/step - accuracy: 0.9889 - loss: 

In [20]:
loss, acc = model.evaluate(test_dataset)

print(f"\nAcurácia final: {acc*100:.2f}%")
print(f"Loss final: {loss:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 291ms/step - accuracy: 0.9615 - loss: 0.0769

Acurácia final: 96.15%
Loss final: 0.0769


In [21]:
model.load_weights('model.weights.best.keras')

In [22]:
def agente_monitoramento(caminho_imagem):

    img = tf.io.read_file(caminho_imagem)

    img = tf.image.decode_image(
        img,
        channels=3
    )

    img = tf.image.resize(
        img,
        image_size
    )

    img = tf.cast(
        img,
        tf.float32
    ) / 255.0

    img = tf.expand_dims(
        img,
        axis=0
    )

    pred = model.predict(
        img,
        verbose=0
    )[0][0]

    if pred > 0.5:

        print(
            f"ALERTA DE ENCHENTE Probabilidade: {pred:.2%}"
        )

    else:

        print(
            f"SEM ENCHENTE Probabilidade: {(1-pred):.2%}"
        )

In [23]:
print("Teste 1")
agente_monitoramento(
    test_image_paths[0]
)

print("\nTeste 2")
agente_monitoramento(
    test_image_paths[10]
)

print("\nTeste 3")
agente_monitoramento(
    test_image_paths[20]
)

Teste 1
SEM ENCHENTE Probabilidade: 99.97%

Teste 2
ALERTA DE ENCHENTE Probabilidade: 99.69%

Teste 3
SEM ENCHENTE Probabilidade: 74.77%


In [24]:
print("\n===== RESULTADOS =====")
print(f"Acurácia: {acc*100:.2f}%")
print(f"Tempo de treinamento: {tempo_total:.2f} segundos")


===== RESULTADOS =====
Acurácia: 96.15%
Tempo de treinamento: 199.17 segundos


# Resultados Obtidos

## Treinamento do Modelo

Foi utilizado o modelo **MobileNetV2** com a técnica de **Transfer Learning**, aproveitando pesos pré-treinados no conjunto ImageNet para realizar a classificação de imagens de enchente.

Foi utilizado o dataset diretamente pois o zip não funcionou.

O melhor modelo encontrado foi salvo automaticamente utilizando o callback `ModelCheckpoint`.

## Avaliação do Modelo

Após o treinamento, o modelo atingiu os seguintes resultados:

- **Acurácia:** 96,15%
- **Tempo de treinamento:** 199,17 segundos

Esses resultados demonstram que o modelo foi capaz de distinguir corretamente imagens com enchentes e imagens normais na grande maioria dos casos.

## Testes do Agente de Monitoramento

Foram realizados testes utilizando imagens do conjunto de teste para validar o funcionamento do agente.

### Teste 1

Resultado:

- Classe prevista: Sem enchente
- Probabilidade: 99,97%


### Teste 2

Resultado:

- Classe prevista: Alerta de enchente
- Probabilidade: 99,69%

### Teste 3

Resultado:

- Classe prevista: Sem enchente
- Probabilidade: 74,77%


## Conclusão



Os resultados indicam que a solução  é capaz de atuar como um agente inteligente de monitoramento, auxiliando na identificação automática de áreas afetadas por enchentes a partir de imagens.